[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/cbm-colab-2026/blob/main/notebooks/2-1_%EC%84%BC%EC%84%9C%EC%8B%A0%ED%98%B8%EC%99%80_%ED%8A%B9%EC%A7%95%EC%B6%94%EC%B6%9C.ipynb)

# 2-1 · 유압 장치 데이터의 구조 — 사이클·원신호·특징, 그리고 좋은 특징 고르기

**⏱ 70분** · 모듈 2: 센서 → **특징** → 이상탐지 → 고장진단 → 판정 기준

## 🎯 이번 시간의 질문

1. 이 데이터는 **무엇을, 어떻게** 기록했고, 파일마다 **한 행은 무엇**인가? (앞 35분)
2. 부품 상태를 알려면 센서의 **무엇을** 봐야 하는가? (뒤 35분)

1-1에서는 파일 하나를 읽고 "한 행 = 디스크 1대의 하루"부터 확인했습니다. 이번 데이터는 **파일이 세 개**이고 모양이 서로 다릅니다. 그래서 먼저 실험을 이해하고, 구조 그림을 본 뒤, **파일을 하나씩 열어** 한 행이 무엇인지 확인합니다. 분석은 그다음입니다.

| 이번 시간 | 코드 |
|---|---|
| ✍️ **직접 입력 (8개)** | 파일 열어 보기 `shape`·`head` (3번) · 상태별 개수 `value_counts` · 원신호 요약 `mean`·`max` · 묶어 요약 `groupby` · 상자그림 · 구분력 |
| ▶ **제공** — 실행하고 **결과만 읽는** 코드 | 실험 일정표, 원신호 그림, 세 표 합치기와 확인, 구분력 함수 |

## 0. 장치와 실험 (읽기, 5분)

독일 ZeMA 연구소의 **유압 시험장치**입니다. 공장의 유압 프레스·사출기와 같은 구성입니다(단순화한 개념도).

```
[주 회로]         전동기 ──▶ 주 펌프 ──▶ 밸브 ──▶ 부하(실린더)      ← 일을 하는 회로
                              │(내부 누설)   │(유로 전환)
                              └──── 축압기 (압력 변동을 받아 줌)
                                     │
[냉각·여과 회로]  기름 탱크 ──▶ 여과 펌프 ──▶ 필터 ──▶ 냉각기 ──▶ 탱크   ← 기름을 식히고 거르는 회로
```

연구자들은 이렇게 실험했습니다.

1. 장치가 **60초짜리 똑같은 작업**(정해진 순서로 압력을 올리고 내리는 일)을 쉬지 않고 반복하게 했습니다. 이 60초 한 번이 **사이클**입니다. 공장 프레스가 제품 하나를 찍는 한 번의 동작과 같은 단위입니다.
2. 반복하는 동안 네 부품의 상태를 정상에서 고장 직전까지 **일부러 조정**했습니다.
3. 사이클마다 **센서 17개의 60초 기록**을 저장하고, 그 사이클의 **부품 상태를 정답으로** 적었습니다. 이렇게 2,205사이클(이어 붙이면 약 37시간)을 기록해 2018년에 공개했습니다.

| 부품 | 열 이름 | 상태 값 (정상 → 고장 직전) | 나빠지면 |
|---|---|---|---|
| 냉각기 | `cooler` | 100 · 20 · 3 (%, 냉각 효율) | 기름이 과열 |
| 밸브 | `valve` | 100 · 90 · 80 · 73 (%, 전환 성능) | 유로 전환이 늦어짐 |
| 펌프 | `pump_leakage` | 0 · 1 · 2 (없음 · 약한 누설 · 심한 누설) | 내보내는 기름이 줄어듦 |
| 축압기 | `accumulator` | 130 · 115 · 100 · 90 (bar, 충전 압력) | 압력 변동을 못 받아 줌 |

## 1. 데이터 구조 한눈에 보기 (읽기, 3분)

연구자가 기록한 두 가지(센서 기록, 정답)가 이 수업의 파일 세 개가 되었습니다.

```
연구자가 기록한 것                          이 수업의 파일                                 한 행       크기
──────────────────────                      ────────────────────────────────────           ─────────   ─────────────────────
사이클마다 부품 상태 (정답)       ─────▶   ① hydraulic_labels.csv       정답 표          사이클 1개   2,205행 ×     8열
사이클마다 센서 17개의 60초 기록  ──┬──▶   ② hydraulic_raw_4sensors.npz 원신호 (4개만)   사이클 1개   2,205행 × 6,000열 (PS1)
                                    │
                                    └──▶   ③ hydraulic_features.csv     특징 표          사이클 1개   2,205행 ×    71열
                                            (17개 센서의 60초 기록을 각각 숫자 5개로 요약)
```

- **세 파일 모두 2,205행이고, i번째 행 = i번째 사이클**입니다. 이 약속 하나로 세 파일이 이어집니다.
- ② **원신호**는 60초 기록을 그대로 담아 **열이 시각**입니다(압력은 1초에 100번 × 60초 = 6,000열).
- ③ **특징**은 원신호 한 행(값 6,000개)을 평균·최대 같은 **숫자 몇 개로 줄인 것**입니다. 분석은 주로 ①과 ③을 붙인 표로 합니다.

이 순서대로 하나씩 열어 봅니다: **① 정답 표 → ② 원신호 → ③ 특징 표 → 세 표 잇기**.

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. (실습 데이터와 한글 글꼴을 내려받습니다)
import os, urllib.request, warnings
REPO = "https://raw.githubusercontent.com/ralbu85/cbm-colab-2026/main/"
for path in ["data/hydraulic_labels.csv",
             "data/hydraulic_features.csv",
             "data/hydraulic_raw_4sensors.npz",
             "fonts/NanumGothic.ttf"]:
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(REPO + path, path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_manager.fontManager.addfont("fonts/NanumGothic.ttf")
plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False,
                     "figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore")
print("준비 완료!")

## 2. ① 정답 표 — 사이클마다 부품 상태

**P1 · 읽고 보기** — 1-1과 똑같이 크기와 앞부분부터 봅니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
labels = pd.read_csv('data/hydraulic_labels.csv')
print(labels.shape)
labels.head()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 2,205행 × 8열입니다. **한 행 = 시험장치 한 대의 사이클 한 번**입니다. 장치 2,205대가 아니라 **한 대를 2,205번** 기록한 것입니다(1-1의 "523만 행 ≠ 523만 대"와 같은 주의).

| 열 | 뜻 |
|---|---|
| `cycle_id` | 사이클 번호 = 기록 순서 (1 ~ 2,205) |
| `cooler`, `valve`, `pump_leakage`, `accumulator` | 그 사이클의 네 부품 상태 — **정답** |
| `stable_flag` | 부품 상태를 바꾼 직후라 장치가 아직 안정되지 않았으면 1 |
| `condition_group` | 네 부품 상태를 이어 붙인 이름. `3_100_0_130` = 냉각기 3 · 밸브 100 · 누설 0 · 축압기 130 |
| `split` | 학습용(train) / 평가용(test) 구분 (2-2부터 사용) |

앞의 다섯 행은 모두 냉각기 3%, 밸브 100%, 누설 없음, 축압기 130 bar입니다. **상태를 바꾸기 전까지 같은 조합이 여러 사이클 이어서** 기록됐기 때문입니다.

**세기** — `value_counts()`는 값마다 몇 행인지 셉니다. 네 부품에 반복합니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
for part in ['cooler', 'valve', 'pump_leakage', 'accumulator']:
    print(labels[part].value_counts().sort_index())
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 냉각기 3%·20%·100%가 약 730사이클씩이고, 축압기 고장 직전(90 bar)은 808사이클로 가장 많습니다. 부품마다 **정상이 아닌 상태가 45~73%**입니다.
1-1의 디스크는 90일 동안 0.33%만 고장 났습니다. 여기에 고장 상태가 이렇게 많은 것은 연구자가 **일부러 상태를 바꿔 가며 기록한 실험**이기 때문입니다. 고장 사례를 골고루 모을 수 있다는 것이 시험장치 데이터의 장점입니다.

연구자가 상태를 **어떤 순서로** 바꿨는지 그려 봅니다(▶ 실험 일정표 — 네 부품 열을 사이클 번호 순서로 그린 것).

In [ ]:
# ▶ 그대로 실행하세요
fig, axes = plt.subplots(4, 1, figsize=(11, 6), sharex=True)
for ax, (part, name) in zip(axes, [('cooler', '냉각기 (%)'), ('valve', '밸브 (%)'),
                                   ('pump_leakage', '펌프 누설'), ('accumulator', '축압기 (bar)')]):
    ax.step(labels['cycle_id'], labels[part], where='post')
    ax.set_ylabel(name, fontsize=9)
axes[-1].set_xlabel('사이클 번호 (시간 순서)')
fig.suptitle('실험 일정표 — 연구자가 부품 상태를 바꾼 순서')
fig.tight_layout()
plt.show()
print('부품 상태 조합:', labels['condition_group'].nunique(), '가지 (= 3 × 4 × 3 × 4)')
print('학습용 / 평가용:', (labels['split'] == 'train').sum(), '/', (labels['split'] == 'test').sum())
print('상태를 바꾼 직후 (stable_flag = 1):', (labels['stable_flag'] == 1).sum())

💬 **결과 해설** — 냉각기는 3% → 20% → 100%의 큰 세 덩어리로, 축압기는 약 730사이클마다 한 바퀴씩, 펌프 누설은 약 40~50사이클, 밸브는 약 10사이클마다 바꿨습니다. 이 실험 방식에서 이 데이터를 읽는 **세 가지 주의점**이 나옵니다.

1. **부품 상태가 섞여 있다** — 네 부품 상태의 조합 144가지를 모두 시험했습니다. "냉각기가 정상인 사이클"에도 누설·밸브 지연·축압기 저압이 섞여 있어서, 한 부품을 볼 때 **다른 부품의 영향이 함께 들어옵니다.**
2. **같은 조합이 연달아 기록됐다** — 이웃한 사이클은 거의 쌍둥이입니다. 무작위로 학습용/평가용을 나누면 시험 문제를 미리 본 셈이 되므로, **같은 조합은 한쪽에만** 넣었습니다(학습 1,869 / 평가 336).
3. **상태를 바꾼 직후는 어중간하다** — 기름 온도 등이 새 상태로 옮겨 가는 중인 사이클이 756개(`stable_flag = 1`)입니다.

## 3. ② 원신호 — 사이클 한 번 동안의 센서 기록

센서마다 1초에 측정하는 횟수가 달라서, 같은 60초라도 값의 개수가 다릅니다.

| 센서 | 측정하는 것 | 단위 | 1초에 측정하는 횟수 | 60초 동안 값의 개수 |
|---|---|---|---|---|
| PS1 ~ PS6 | 압력 (6곳) | bar | 100번 | 6,000개 |
| EPS1 | 전동기 전력 | W | 100번 | 6,000개 |
| FS1, FS2 | 유량 (2곳) | L/min | 10번 | 600개 |
| TS1 ~ TS4 | 기름 온도 (4곳) | °C | 1번 | 60개 |
| VS1 | 진동 | mm/s | 1번 | 60개 |

원래 공개 데이터는 **센서마다 파일이 하나씩**(17개) 있습니다. 이 수업의 `.npz` 파일에는 그중 **4개(PS1·FS1·TS1·VS1)**를 담았습니다. `.npz`는 여러 개의 표(배열)를 한 파일에 묶은 것으로, `raw['PS1']`처럼 이름으로 꺼냅니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
raw = np.load('data/hydraulic_raw_4sensors.npz')
PS1 = raw['PS1']
print(PS1.shape)
print(raw['FS1'].shape, raw['TS1'].shape, raw['VS1'].shape)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — `(2205, 6000)`, `(2205, 600) (2205, 60) (2205, 60)`.

💬 **결과 해설** — 원신호도 **2,205행 = 사이클 2,205개**이고 순서는 정답 표와 같습니다. 다른 점은 **열이 시각**이라는 것입니다.

```
            열 0      열 1      열 2     …   열 5999
            (0.00초)  (0.01초)  (0.02초)      (59.99초)
PS1[0]      151.47    151.45    151.52   …   151.19     ← 사이클 1 (정답 표 cycle_id 1)
PS1[1]      151.11    151.12    151.16   …   …          ← 사이클 2
  ⋮
PS1[2204]   151.83    151.78    …                       ← 사이클 2,205
```

- 압력은 1초에 100번이므로 **열 번호 ÷ 100 = 사이클 안의 초**입니다. 유량(600열)은 ÷ 10, 온도·진동(60열)은 열 번호가 곧 초입니다.
- 파이썬은 0부터 세므로 **사이클 c = `PS1[c − 1]`**입니다(사이클 1 = `PS1[0]`).

한 행을 그리면 사이클 한 번 동안의 압력 변화가 됩니다. 0·1·2번 행을 이어 붙여 **180초**를 그려 봅니다(▶ 그림만 읽으세요).

In [ ]:
# ▶ 그대로 실행하세요
t3 = np.arange(3 * 6000) / 100                              # 0 ~ 180초 (1초에 100번 측정)
plt.figure(figsize=(12, 3.5))
plt.plot(t3, np.concatenate([PS1[0], PS1[1], PS1[2]]))
for boundary in [60, 120]:
    plt.axvline(boundary, color='red', linestyle='--')
for i in range(3):
    plt.text(i * 60 + 30, 196, f'PS1[{i}] = 사이클 {i + 1}', ha='center', fontsize=11)
plt.ylim(140, 200)
plt.xlabel('시간 (초)')
plt.ylabel('압력 PS1 (bar)')
plt.title('원신호 한 행 = 60초짜리 작업 한 번 → 같은 모양이 반복된다')
plt.show()

💬 **결과 해설** — 60초마다 **같은 모양**이 반복됩니다. 한 사이클 안에서 압력은 10초마다 계단처럼 바뀝니다. 장치가 정해진 순서로 일을 바꾸기 때문입니다(첫 1~2초는 직전 단계의 151 bar에서 191 bar로 올라가는 중).

| 사이클 안의 시간 | 0~10초 | 10~20초 | 20~30초 | 30~40초 | 40~50초 | 50~60초 |
|---|---|---|---|---|---|---|
| 압력 PS1 (약) | **191 bar** | 146 | 156 | 166 | 156 | 151 |

같은 사이클 1을 다른 센서로 보면 이렇습니다(▶ 원신호 네 배열의 0번 행).

In [ ]:
# ▶ 그대로 실행하세요
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, (name, key, hz, unit) in zip(axes, [('PS1 압력', 'PS1', 100, 'bar'), ('FS1 유량', 'FS1', 10, 'L/min'),
                                            ('TS1 온도', 'TS1', 1, '°C'), ('VS1 진동', 'VS1', 1, 'mm/s')]):
    signal = raw[key][0]
    ax.plot(np.arange(len(signal)) / hz, signal)
    ax.set_title(f'{name} ({len(signal)}개)'); ax.set_xlabel('사이클 안의 시간 (초)'); ax.set_ylabel(unit)
fig.tight_layout()
plt.show()

💬 같은 60초인데 값의 개수가 6,000 / 600 / 60 / 60개입니다. 압력·유량은 작업 단계에 따라 계단처럼 바뀌고, 온도는 거의 일정합니다. 이렇게 **모양이 다른 원신호를 그대로 비교하기는 어렵습니다.** 그래서 행마다 숫자 몇 개로 요약한 것이 ③ 특징 표입니다.

## 4. ③ 특징 표 — 원신호 한 행을 숫자 몇 개로

**P1 · 읽고 보기**

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
features = pd.read_csv('data/hydraulic_features.csv')
print(features.shape)
features.head()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 2,205행 × 71열입니다. **한 행 = 사이클 1번**이고, 열은 `cycle_id` + **센서 17개 × 요약 5개 = 70열**입니다(열이 많아 가운데는 `...`로 줄여 보입니다). 이름은 `센서_요약` 형식입니다.

| 요약 | 뜻 | 사이클 1의 PS1 |
|---|---|---|
| `_mean` | 60초 평균 — 전체 수준 | 160.67 |
| `_std` | 표준편차 — 얼마나 흔들렸나 | 13.94 |
| `_min`, `_max` | 가장 작은 값, 가장 큰 값 | 145.83, 191.51 |
| `_change` | 마지막 1초 평균 − 첫 1초 평균 — 사이클 동안 얼마나 변했나 | −25.76 |

②에는 센서 4개만 있지만, ③은 원래 공개된 17개 센서 파일에서 **모두** 계산해 두었습니다.

**특징이 정말 원신호의 요약인지** 직접 확인합니다. ② 원신호의 첫 행(사이클 1의 압력 6,000개)을 요약해 ③의 첫 행과 비교합니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
x = PS1[0]                      # 사이클 1의 압력 6,000개
print(x.mean(), x.max())
print(features['PS1_mean'][0], features['PS1_max'][0])
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 두 줄 모두 평균 160.67, 최대 191.51.

💬 특징 표의 `PS1_mean`은 **사이클마다 원신호 한 행(6,000개)의 평균**을 미리 계산해 둔 것입니다. 이제부터 사이클 하나를 **숫자 70개**로 봅니다.

## 5. 세 표 잇기 — 사이클 번호 하나로 (▶)

정답 표와 특징 표를 `cycle_id`가 같은 행끼리 **옆으로 붙여** 표 하나(`data`)로 만듭니다(엑셀의 VLOOKUP과 같은 일). 그리고 **사이클 1000번**을 세 곳에서 꺼내 정말 같은 사이클인지 확인합니다.

In [ ]:
# ▶ 그대로 실행하세요
data = labels.merge(features, on='cycle_id')     # cycle_id가 같은 행끼리 옆으로 붙이기
print('합친 표:', data.shape, '(정답 8열 + 특징 70열, cycle_id는 한 번만)')
print()
c = 1000
print('① 정답 표의 1000번 사이클')
print(labels[labels['cycle_id'] == c][['cycle_id', 'cooler', 'valve', 'pump_leakage', 'accumulator']].to_string(index=False))
print('② 원신호 PS1[999] (= 1000번째 행)의 평균 :', round(PS1[c - 1].mean(), 2))
print('③ 특징 표 1000번 사이클의 PS1_mean     :', round(features.loc[features['cycle_id'] == c, 'PS1_mean'].values[0], 2))

💬 **결과 해설** — 합친 표는 2,205행 × 78열입니다. **한 행 = 사이클 1번**, 왼쪽 열은 **정답**(부품 상태), 오른쪽 70열은 **입력**(센서 특징)입니다. 앞으로 모듈 2의 분석은 이 표로 합니다.
1000번 사이클은 냉각기 20%, 밸브 80%, 약한 누설, 축압기 130 bar 상태였고, 그때의 압력 평균은 원신호에서 계산해도(②) 특징 표에서 읽어도(③) 158.29 bar로 같습니다. ② 원신호는 표에 붙이지 않고, 필요할 때 `PS1[c - 1]`로 꺼내 씁니다(과제 3).

## 6. 좋은 특징이란 — 상자그림과 구분력

이제 두 번째 질문입니다. 1-1에서 모델별로 고장률을 비교했듯이, **냉각기 상태별로** 오일 온도(`TS1_mean`)를 비교합니다.

**P3 · 묶어 요약**

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
data.groupby('cooler')['TS1_mean'].mean()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


**P4 · 상자그림** — 평균 하나로는 흩어짐을 알 수 없으므로, 상태별로 값의 범위(상자)와 가운데(선)를 봅니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
data.boxplot(column='TS1_mean', by='cooler')
plt.suptitle('')          # pandas가 자동으로 붙이는 윗제목 지우기
plt.xlabel('냉각기 상태 (%)')
plt.ylabel('오일 온도 TS1 평균 (°C)')
plt.show()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 냉각기가 나빠질수록(100 → 20 → 3) 오일 온도가 약 **36 → 45 → 55°C**로 오르고, 세 상자가 겹치지 않습니다. 냉각이 안 되면 기름이 뜨거워지는 물리 관계 그대로입니다. 상자 밖의 점 56개는 모두 2절의 "상태를 바꾼 직후"(`stable_flag = 1`) 사이클입니다.

### 구분력 — 특징이 얼마나 좋은지를 숫자 하나로

센서마다 단위와 크기가 다릅니다(압력 약 160 bar, 전력 약 2,500 W, 유량 약 6 L/min). 그래서 값의 차이를 그대로 비교하지 않고, "**상자 사이의 거리가 상자 크기의 몇 배인가**"로 비교합니다.

$$\text{구분력} = \frac{\text{상태별 평균 중 가장 큰 값} - \text{가장 작은 값}}{\text{상태 안 표준편차의 평균}}$$

| 구분력 | 의미 |
|---|---|
| 3 이상 | 상자가 거의 겹치지 않음 → 특징 하나로 상태를 거의 알 수 있음 |
| 1 근처 | 상자가 많이 겹침 → 특징 하나로는 부족 |

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
g = data.groupby('cooler')['TS1_mean']
print((g.mean().max() - g.mean().min()) / g.std().mean())
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 약 **11.4**. 가장 먼 두 상태의 온도 차이(19.0°C)가 상태 안 흩어짐(평균 1.7°C)의 11배라서 상자가 겹치지 않습니다.

▶ 같은 계산을 70개 특징 전부에 반복하는 함수입니다. `rank_features(부품)` → 구분력이 높은 특징 5개.

In [ ]:
# ▶ 그대로 실행하세요
def separation(part, feature):
    """구분력 = (상태별 평균의 최댓값 − 최솟값) ÷ (상태 안 표준편차의 평균)."""
    g = data.groupby(part)[feature]
    return round((g.mean().max() - g.mean().min()) / g.std().mean(), 2)

def rank_features(part, top=5):
    """70개 센서 특징의 구분력을 높은 순서로 보여 준다."""
    cols = [c for c in features.columns if c != 'cycle_id']
    return pd.Series({c: separation(part, c) for c in cols}).sort_values(ascending=False).head(top)

rank_features('cooler')

💬 냉각기에는 구분력 13 안팎의 특징이 여럿 있습니다(온도 TS4, 압력 PS5·PS6 — 기름이 뜨거워지면 묽어져 압력도 달라지기 때문으로 보임). **냉각기는 "좋은 특징"이 많은 부품**입니다. 다른 부품도 그럴까요? → 과제 2.

---
## 7. 실습 과제 (25분)

### 📝 과제 1 · 압력 PS4의 0은 무엇일까? (데이터 파악)
압력 센서 PS4의 60초 평균(`PS4_mean`)은 0인 사이클이 많습니다. **몇 개**인지 세고, **냉각기 상태와 관계가 있는지** 교차표(1-2에서 쓴 `pd.crosstab`)로 확인한 뒤, 이 0을 어떻게 다뤄야 할지 적으세요.

| 냉각기 상태 | PS4 = 0인 사이클 | 전체 |
|---|---|---|
| 3% | | 732 |
| 20% | | 732 |
| 100% | | 741 |

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트 1: (data['PS4_mean'] == 0).sum()  — 조건이 참인 개수
# 힌트 2: pd.crosstab(data['cooler'], data['PS4_mean'] == 0)


✏️ **나의 답:**

### 📝 과제 2 · 부품별로 가장 좋은 특징 찾기
`rank_features()`로 나머지 세 부품의 1위 특징과 구분력을 찾아 표를 채우세요. **실행 전에** 어느 부품이 가장 어려울지 예상해 보세요.
시간이 남으면 펌프의 1위 특징을 6절처럼 상자그림으로 그려 겹침을 눈으로 확인하세요.

| 부품 | 1위 특징 | 구분력 | 특징 하나로 충분한가? |
|---|---|---|---|
| 냉각기 `cooler` | PS6_min | 13.23 | 예 |
| 펌프 `pump_leakage` | | | |
| 밸브 `valve` | | | |
| 축압기 `accumulator` | | | |

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: rank_features('pump_leakage') 처럼 부품 이름만 바꿉니다.
# 상자그림: 6절 코드에서 'TS1_mean' → 1위 특징, 'cooler' → 'pump_leakage'


✏️ **나의 답:**

### 📝 과제 3 · 밸브 특징 새로 만들기 (도전)
밸브는 1위 특징도 구분력 1.2로 약했습니다. 60초 전체를 요약한 특징이 놓친 것이 있을 수 있습니다. 이번에는 ② **원신호**를 직접 씁니다.

1. **다른 부품은 모두 정상이고 밸브만 다른** 네 사이클의 압력을 겹쳐 그려 **차이 나는 시간대**를 찾으세요: 사이클 1788(밸브 100), 1778(90), 1768(80), 1758(73). 사이클 c = `PS1[c − 1]`이므로 사이클 1788은 `PS1[1787]`, 시간 축은 `t = np.arange(6000) / 100`.
2. 찾은 구간의 압력 평균을 새 특징으로 만들고 구분력을 비교하세요. 9~10초는 열 900~999이므로 `PS1[:, 900:1000]`(모든 행의 900~999열), 행마다 평균은 `.mean(axis=1)`입니다.

| 밸브 진단용 특징 | 구분력 |
|---|---|
| FS1_max (기존 1위) | 1.18 |
| PS1_peak9 (9~10초 압력 평균) | |

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트 1: plt.plot(t, PS1[1787], label='밸브 100') 을 네 번 부르고 plt.xlim(8, 12) 로 확대
# 힌트 2: data['PS1_peak9'] = PS1[:, 900:1000].mean(axis=1)  →  separation('valve', 'PS1_peak9')


✏️ **나의 답:**

---
## ✅ 정리

| 배운 것 | 핵심 |
|---|---|
| 실험 | 60초 작업(사이클)을 반복하며 연구자가 네 부품 상태를 **일부러** 바꾸고, 사이클마다 센서 기록 + 정답을 남김 |
| 데이터 구조 | 세 파일 모두 **한 행 = 사이클 1번**, i번째 행 = i번째 사이클. ① 정답 2,205 × 8 · ② 원신호 2,205 × 6,000(열 = 시각) · ③ 특징 2,205 × 71 |
| 원신호 → 특징 | 원신호 한 행(6,000개)을 평균·최대 등으로 요약 = 특징 표 한 행. 사이클 c의 원신호는 `PS1[c - 1]` |
| 세 표 잇기 | `labels.merge(features, on='cycle_id')` → 2,205 × 78. 왼쪽은 정답, 오른쪽은 입력 |
| 실험 설계에서 나온 주의점 | 부품 상태가 섞인다 · 같은 조합이 연달아 → 조합 단위로 학습/평가 분리 · 상태 변경 직후는 과도 구간 |
| 구분력 | 상자 사이 거리 ÷ 상자 크기. 냉각기 13 · 펌프 1.1 · 밸브 1.2 · 축압기 1.2 → **냉각기만 쉬운 부품** |
| 데이터 의심하기 | 빈칸이 없어도 원인 모를 값이 있다 (PS4의 0) |

다음 질문: 구분력이 높은 냉각기는 **정상 데이터만으로도** 경보를 만들 수 있을까요? 구분력이 1밖에 안 되는 펌프 누설은요? → **2-2 · 이상탐지**